In [1]:
print('hi')

hi


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 뉴스 요약 함수 (mock)
def get_news_summary(topic: str) -> str:
    summaries = {
        "ai_trends": "2025년 AI 트렌드는 멀티모달 모델과 AI 에이전트의 급성장이 핵심입니다. GPT-5, Gemini Ultra 등 대형 모델들이 경쟁하고 있습니다.",
        "machine_learning": "머신러닝 분야에서는 소형 언어 모델(SLM)이 주목받고 있습니다. 적은 자원으로도 높은 성능을 내는 모델 경량화 기술이 발전하고 있습니다.",
        "ai_jobs": "AI로 인한 일자리 변화가 가속화되고 있습니다. 반복 업무는 자동화되지만, AI를 다루는 새로운 직업군이 빠르게 성장하고 있습니다.",
        "ai_ethics": "AI 윤리 규제가 전 세계적으로 강화되고 있습니다. EU AI Act가 시행되며 투명성과 책임성이 중요한 이슈로 떠오르고 있습니다.",
        "ai_healthcare": "의료 AI 분야에서는 암 조기 진단 정확도가 인간 의사 수준에 도달했습니다. AI 신약 개발 속도도 크게 단축되고 있습니다.",
    }
    return summaries.get(topic, f"{topic}에 대한 뉴스를 찾을 수 없습니다.")

tool_map = {"get_news_summary": get_news_summary}

tools = [
    {
        "type": "function",
        "name": "get_news_summary",
        "description": "특정 AI 관련 토픽의 최신 뉴스 요약을 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "뉴스를 조회할 토픽 (예: ai_trends, machine_learning, ai_jobs)"
                }
            },
            "required": ["topic"]
        }
    }
]

input_msg = [{'role': 'user', 'content': 'AI 트렌드, 머신러닝, AI 일자리, AI 윤리, 의료 AI 뉴스를 각각 요약해줘.'}]

# 1차 호출
resp1 = client.responses.create(
    model='gpt-4o-mini',
    input=input_msg,
    tools=tools,
    tool_choice='auto'
)

# 함수 실행
tool_outputs = []
news_data = {}
for item in resp1.output:
    if item.type == 'function_call':
        args = json.loads(item.arguments)
        result = tool_map[item.name](**args)
        news_data[args['topic']] = result
        tool_outputs.append({
            'type': 'function_call_output',
            'call_id': item.call_id,
            'output': result
        })

print('수집된 뉴스 토픽:', list(news_data.keys()))


In [ ]:
from docx import Document

doc = Document()

for key, value in news_data.items():
    heading = ' '.join(word.capitalize() for word in key.split('_'))
    doc.add_heading(heading, level=1)
    doc.add_paragraph(value)
    doc.add_paragraph()  # 빈 줄 추가

doc.save('test.docx')
print('test.docx 저장 완료!')


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
default_model = os.getenv('OPENAI_DEFAULT_MODEL', 'gpt-4o-mini')

response = client.responses.create(
    model=default_model,
    input=[
        {'role': 'system', 'content': '주어진 텍스트를 이모지로 변환해줘'},
        {'role': 'user', 'content': '졸립고, 덥고, 짜증나고, 무슨 소린지 못 알아듣겠고, 강사는 끊임 없이 떠들고, 집에 가고 싶다'}
    ]
)
print(response.output_text)


In [ ]:
import gradio as gr

def func(name):
        return f"Hello, {name}!"

demo= gr.Interface(func, "text", "text").launch()

demo.launch()
